# Lab 5 · Phân khúc giá & host chuyên nghiệp (groupby, transform, merge)

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · bài 5**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo bài 5 phân tích theo **quận**. Lab này phân tích thị trường theo hai chiều khác:
**phân khúc giá** (rẻ / trung / cao) và **kiểu host** (cá nhân / chuyên nghiệp), đồng thời thực hành
`map` → `groupby`/`agg` → `transform` → `pivot_table` → `merge`.

## Cách làm việc trong buổi lab

- Bài tập được chia thành từng bước; mỗi bước có ô `TODO` và phần kiểm tra `assert`.
  Hoàn thành toàn bộ `assert` nghĩa là kết quả đáp ứng yêu cầu.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🚫 ở giờ lý thuyết đánh giá các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn ✅ mở: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Nếu chưa giải quyết được một bước sau 3 phút, hãy gọi giảng viên thực hành đến hỗ trợ.

## Mục tiêu

Sau buổi lab, bạn sẽ:

1. Dùng `map` với **hàm** để sinh cột phân loại từ cột số.
2. Dùng `groupby` + `agg` đặt tên trên nhiều cột và diễn giải kết quả.
3. Dùng `transform` để gắn thông tin nhóm về từng dòng.
4. Tạo `pivot_table` hai chiều và `merge` bảng tra cứu có `validate`.

## Phần 0 · Khởi động (~10 phút)

In [1]:
import pandas as pd

# W1 — loc theo nhãn, iloc theo vị trí
s = pd.Series([100, 200, 300], index=[5, 0, 2])

# TODO: điền hai giá trị
theo_nhan = s.loc[2]      # phần tử mang NHÃN 2
theo_vi_tri = s.iloc[2]   # phần tử ở VỊ TRÍ 2 (thứ ba)

# --- Ô kiểm tra ---
assert theo_nhan == 300 and theo_vi_tri == 300 or True  # xem giải thích dưới
assert s.loc[0] == 200 and s.iloc[0] == 100
assert theo_nhan == s.loc[2] and theo_vi_tri == s.iloc[2]
print("W1 ổn: loc[2] =", theo_nhan, "· iloc[2] =", theo_vi_tri)

W1 ổn: loc[2] = 300 · iloc[2] = 300


In [2]:
from numpy import nan
# W2 — alignment: phép toán tự khớp theo index
gia_cu = pd.Series({"A": 100, "B": 200, "C": 50})
gia_moi = pd.Series({"B": 220, "A": 90, "D": 70})

# TODO: tính thay đổi tuyệt đối (gia_moi - gia_cu) và đếm số NaN trong kết quả
thay_doi = gia_moi - gia_cu
so_nan = thay_doi.isna().sum()

# --- Ô kiểm tra ---
assert thay_doi["A"] == -10 and thay_doi["B"] == 20
assert so_nan == 2
print("W2 ổn: pandas khớp theo NHÃN (A với A, B với B) — C và D lệch nhau thành NaN.")

W2 ổn: pandas khớp theo NHÃN (A với A, B với B) — C và D lệch nhau thành NaN.


## Phần 1 · Phân tích thị trường theo hai chiều (~55 phút)

In [3]:
URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/visualisations/listings.csv")
df = pd.read_csv(URL)
len(df)

18534

### Bước 1 · Sinh cột phân khúc bằng map với hàm

Chia giá thành 3 phân khúc: `re` (< 30.000), `trung` (30.000–99.999), `cao` (≥ 100.000);
giá thiếu → `None`. Logic có rẽ nhánh nên `map` với hàm phù hợp hơn phép toán vector hoá
trong trường hợp này.

In [4]:
def phan_khuc(gia):
    # TODO: viết thân hàm theo mô tả trên (nhớ xử lý NaN bằng pd.isna trước)
    if pd.isna(gia):
      return None
    elif gia < 30_000:
      return "re"
    elif gia < 99_999:
      return "trung"
    elif gia >= 100_000:
      return "cao"
    else:
      return None

df["phan_khuc"] = df["price"].map(phan_khuc)
dem = df["phan_khuc"].value_counts(dropna=False)

# --- Ô kiểm tra ---
assert dem["trung"] == 12325 and dem["cao"] == 3703 and dem["re"] == 1660
print(dem)

phan_khuc
trung    12325
cao       3703
re        1660
None       846
Name: count, dtype: int64


### Bước 2 · agg đặt tên — thống kê ba phân khúc

`number_of_reviews_ltm` là số review trong 12 tháng qua, được dùng làm chỉ báo gần đúng (proxy) cho "mức độ được đặt".

In [5]:
# TODO: groupby phan_khuc, agg 3 cột đặt tên:
#   so_phong   = ("id", "size")
#   gia_trung_vi = ("price", "median")
#   review_nam_tb = ("number_of_reviews_ltm", "mean")
tk = df.groupby("phan_khuc").agg(
  so_phong   = ("id", "size"),
  gia_trung_vi = ("price", "median"),
  review_nam_tb = ("number_of_reviews_ltm", "mean")

)

# --- Ô kiểm tra ---
assert tk.loc["trung", "so_phong"] == 12325
assert tk.loc["cao", "gia_trung_vi"] == 145318.0
assert round(tk.loc["trung", "review_nam_tb"], 1) == 15.4
tk.round(1)

,so_phong,gia_trung_vi,review_nam_tb
phan_khuc,,,
cao,3703,145318.0,11.2
re,1660,23617.5,10.0
trung,12325,54776.0,15.4


Đọc bảng: phân khúc **trung** có nhiều phòng nhất và số review trung bình cao nhất
(15,4 review/năm). Phân khúc **cao** có giá gần gấp ba lần nhưng số review trung bình thấp hơn.
Bảng `agg` ba dòng cho phép so sánh trực tiếp các phân khúc.

### Bước 3 · Host chuyên nghiệp

Cột `calculated_host_listings_count` cho biết host của phòng đang có bao nhiêu listing.
Quy ước host có **≥ 5 listing** là "chuyên nghiệp".

In [6]:
# TODO: tạo cột bool df["chuyen"]; tính tỷ lệ phòng thuộc host chuyên nghiệp
df["chuyen"] = df["calculated_host_listings_count"] >= 5
ty_le_chuyen = df["chuyen"].mean()

# TODO: bảng so sánh 2 nhóm chuyen (groupby "chuyen"), 2 cột đặt tên:
#   gia_trung_vi=("price", "median") · review_nam_tb=("number_of_reviews_ltm", "mean")
so_sanh = df.groupby("chuyen").agg(
    gia_trung_vi = ("price","median"),
    review_nam_tb = ("number_of_reviews_ltm","mean")
)

# --- Ô kiểm tra ---
assert round(ty_le_chuyen, 3) == 0.358
assert so_sanh.loc[True, "gia_trung_vi"] == 65078.0
assert round(so_sanh.loc[False, "review_nam_tb"], 1) == 12.7
so_sanh.round(1)

,gia_trung_vi,review_nam_tb
chuyen,,
False,55589.0,12.7
True,65078.0,15.5


36% số listing thuộc các host chuyên nghiệp. Nhóm này có giá trung vị cao hơn khoảng 17%
và số review trung bình cũng cao hơn. Câu hỏi: *các số liệu này đã đủ để kết luận host
chuyên nghiệp hoạt động hiệu quả hơn chưa, hay còn biến ẩn?* (gợi ý: họ tập trung ở quận nào?)

### Bước 4 · transform — gắn quy mô quận vào từng dòng

`transform` trả kết quả **đúng độ dài bảng gốc**. Dùng nó để gắn quy mô quận vào mỗi dòng,
sau đó lọc các quận quá nhỏ trước khi so sánh.

In [7]:
# TODO: tạo cột n_quan = cỡ quận (transform "size" trên groupby neighbourhood)
df["n_quan"] = df.groupby("neighbourhood").transform("size")
# TODO: lọc df_lon = các dòng thuộc quận có >= 300 phòng
df_lon = df[df["n_quan"] >= 300]

# --- Ô kiểm tra ---
assert len(df_lon) == 16850
assert df_lon["neighbourhood"].nunique() == 8
print(f"{len(df_lon):,} phòng thuộc {df_lon['neighbourhood'].nunique()} quận đủ lớn.")

16,850 phòng thuộc 8 quận đủ lớn.


### Bước 5 · pivot_table — phân khúc × kiểu host

In [8]:
# TODO: pivot_table trên df: index=phan_khuc, columns=chuyen,
#       values=number_of_reviews_ltm, aggfunc="mean"
pv = df.pivot_table(values="number_of_reviews_ltm",index="phan_khuc",columns="chuyen",aggfunc="mean")

# --- Ô kiểm tra ---
assert round(pv.loc["trung", True], 1) == 16.5
assert round(pv.loc["cao", False], 1) == 9.5
pv.round(1)

chuyen,False,True
phan_khuc,,
cao,9.5,13.5
re,9.2,13.1
trung,14.8,16.5


Đọc bảng chéo: ở **mọi** phân khúc, host chuyên nghiệp đều có số review trung bình cao hơn;
chênh lệch lớn nhất ở phân khúc cao (13,5 so với 9,5). Kết quả ở Bước 3 vẫn xuất hiện khi
tách theo phân khúc. Đây chưa phải bằng chứng về quan hệ nhân quả.

### Bước 6 · merge bảng tra cứu + validate

Giả sử công ty áp **phí dịch vụ theo phân khúc**: rẻ 10%, trung 13%, cao 15%.
Bảng tra cứu gồm 3 dòng và được ghép vào bảng có 18.534 dòng.

In [15]:
phi = pd.DataFrame({"phan_khuc": ["re", "trung", "cao"],
                    "phi_dich_vu": [0.10, 0.13, 0.15]})

# TODO: merge df với phi theo phan_khuc, how="left", validate="m:1"
m = df.merge(phi,on="phan_khuc",how="left",validate="m:1")

# Thói quen sau merge: số dòng + độ khớp
# TODO: đếm số dòng của m và số NaN của cột phi_dich_vu
so_dong_m = len(m)
so_nan_phi = m["phi_dich_vu"].isna().sum()

# --- Ô kiểm tra ---
assert so_dong_m == 18534, "merge làm đổi số dòng?!"
assert so_nan_phi == 846, "NaN đúng bằng số phòng không có phân khúc (không giá)"
m["gia_tong"] = m["price"] * (1 + m["phi_dich_vu"])
gia_tong_tv = m.groupby("phan_khuc")["gia_tong"].median().round(0)
assert gia_tong_tv["trung"] == 61897.0
gia_tong_tv

,gia_tong
phan_khuc,
cao,167116.0
re,25979.0
trung,61897.0


846 giá trị NaN ở cột phí tương ứng với các phòng không có giá nên chưa được gán phân khúc.
Sau merge, cần giải thích các giá trị NaN để đánh giá độ khớp của hai bảng.

## Phần 2 · Bài tự làm ✅ mở (làm sớm tại lớp hoặc làm tại nhà)

Bạn được dùng AI theo quy trình 5 bước; hãy ghi lại prompt và cách kiểm chứng.

### Tự làm 1 · Phòng đắt nhất từng quận

Với `df_lon` (8 quận lớn): dùng `transform("max")` trên giá theo quận để tìm **phòng đắt
nhất của từng quận** bằng cách so sánh `price` với giá lớn nhất của quận. In các cột
`name`, `neighbourhood` và `price`.
Kiểm chứng: giá lớn nhất của quận Santiago phải là 97.000.045 CLP.

### Tự làm 2 · Cơ cấu phân khúc theo quận

Dùng `pivot_table` (hoặc `groupby` + `value_counts(normalize=True)`) tính **tỷ trọng
3 phân khúc trong từng quận lớn**. Quận nào có tỷ trọng phân khúc cao lớn nhất? Kết quả có
khớp với bảng xếp hạng trung vị của lab 2 (Lo Barnechea, Vitacura dẫn đầu) không?

In [ ]:
# bài tự làm 1


In [ ]:
# bài tự làm 2

## Tóm tắt buổi lab

| Nội dung chính | Sẽ gặp lại ở |
|---|---|
| `map` với hàm sinh cột phân loại | tạo biến phân loại từ dữ liệu số |
| `agg` đặt tên nhiều cột — đọc bảng thành nhận xét | xây dựng bảng tổng hợp và KPI |
| `transform` gắn thông tin nhóm về dòng (`size`, `max`) | kiểm tra ngoại lai theo nhóm (bài 10) |
| `pivot_table` kiểm tra kết luận theo biến phân nhóm khác | phân tích bảng chéo (bài 14) |
| `merge` + `validate` + giải thích từng NaN | ghép bảng vùng hoặc tỷ giá |

Buổi lý thuyết tiếp theo: làm việc với dữ liệu từ API, SQL và DuckDB.